In [1]:
!pip install wfdb resampy ishneholterlib

In [ ]:
if not os.path.exists("/content/CMED-Mini-Project"):
    !git clone https://github.com/Anonymous0002396/CMED-Mini-Project.git /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

Cloning into 'CMED-Mini-Project'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 116 (delta 5), reused 114 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 9.51 MiB | 30.24 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [4]:
import sys
from pathlib import Path

project_root = Path("/content/CMED-Mini-Project")
repo_root = project_root / "SSSD-ECG"

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "ptb_xl"))
sys.path.insert(0, str(repo_root / "src" / "sssd"))

print(project_root)
print(repo_root.exists())

/content/CMED-Mini-Project
True


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

from pathlib import Path

In [5]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

if not os.path.exists('/content/synthetic_ptbxl'): # Replace with the actual folder name inside the zip
    !cp "/content/drive/MyDrive/Dataset.zip" /content/synthetic_ptbxl.zip
    !unzip -oq /content/synthetic_ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("Dataset already exists, skipping extraction.")

Extraction complete.


In [21]:
if not os.path.exists('/content/ptbxl'): # Replace with the actual folder name inside the zip
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip
    !unzip -oq /content/ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("Dataset already exists, skipping extraction.")

Extraction complete.


### Set up paths

In [6]:
target_fs = 100

# raw real PTB-XL folder you just extracted
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")

# processed output folder that load_dataset(...) will read from later
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

print("raw exists:", data_folder_ptb_xl.exists())
print("processed exists:", target_folder_ptb_xl.exists())

raw exists: True
processed exists: True


### Real PTB-XL preprocessing

In [7]:
df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

print(df_ptb_xl.shape)
print(type(lbl_itos_ptb_xl))
print(target_folder_ptb_xl)

  0%|          | 0/21799 [00:00<?, ?it/s]

(21799, 44)
<class 'dict'>
/content/processed_ptb_xl_fs100


In [8]:
reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / "memmap.npy",
    data_folder=target_folder_ptb_xl,
    delete_npys=True
)

  0%|          | 0/21799 [00:00<?, ?it/s]

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,label_diag_numeric,label_form_numeric,label_rhythm_numeric,label_diag_subclass_numeric,label_diag_superclass_numeric,data,data_mean,data_std,data_length,data_original
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,[37],[7],[7],[14],[3],0,"[0.0015079858, 0.0007229996, -0.0017089996, 0....","[0.1090222, 0.083223335, 0.11311742, 0.2142296...",1000,00001_lr.npy
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,[37],[],[6],[14],[3],1,"[-0.0021810099, -0.00016502624, -0.00040496505...","[0.13342775, 0.24974498, 0.23233838, 0.4500496...",1000,00002_lr.npy
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],2,"[-0.0027700001, 0.005145004, -0.00023700125, 0...","[0.12119952, 0.14438936, 0.11632309, 0.1476423...",1000,00003_lr.npy
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],3,"[-0.003263996, 0.003501991, -0.00078800163, -0...","[0.13780256, 0.43816054, 0.18206717, 0.4213087...",1000,00004_lr.npy
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,[37],[],[7],[14],[3],4,"[-0.003769, -0.003054002, 0.0016899869, 2.3997...","[0.08270639, 0.23678097, 0.14218302, 0.3419054...",1000,00005_lr.npy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,[36],"[8, 13, 18]",[8],[20],[4],21794,"[-0.010336004, -0.0058109984, -0.0007689993, -...","[0.21377766, 0.22204638, 0.13152157, 0.2788991...",1000,21833_lr.npy
21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,[37],[0],[7],[14],[3],21795,"[-0.0024299999, -0.0038730002, -0.0013400039, ...","[0.09910399, 0.08042687, 0.0962622, 0.19413967...",1000,21834_lr.npy
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,[24],[],[7],[6],[4],21796,"[0.0005970012, -0.0027460016, -0.0006459985, -...","[0.096423626, 0.10434124, 0.13238312, 0.167256...",1000,21835_lr.npy


In [10]:
df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print(type(df_mapped), df_mapped.shape)
print(type(lbl_itos_dict))
print(lbl_itos_dict.keys())

<class 'pandas.DataFrame'> (21799, 45)
<class 'dict'>
dict_keys(['label_all', 'label_diag', 'label_form', 'label_rhythm', 'label_diag_subclass', 'label_diag_superclass'])


In [14]:
label_key = "label_all" 
label_names = np.array(lbl_itos_dict[label_key])

print(label_key, len(label_names))
print(label_names[:30])

label_all 71
['1AVB' '2AVB' '3AVB' 'ABQRS' 'AFIB' 'AFLT' 'ALMI' 'AMI' 'ANEUR' 'ASMI'
 'BIGU' 'CLBBB' 'CRBBB' 'DIG' 'EL' 'HVOLT' 'ILBBB' 'ILMI' 'IMI' 'INJAL'
 'INJAS' 'INJIL' 'INJIN' 'INJLA' 'INVT' 'IPLMI' 'IPMI' 'IRBBB' 'ISCAL'
 'ISCAN']


### Load synthetic dataset

In [16]:
# released synthetic PTB-XL dataset root after unzip
synthetic_root = Path("/content/Dataset")

syn_train_data_path = synthetic_root / "data" / "ptbxl_train_data.npy"
syn_val_data_path   = synthetic_root / "data" / "ptbxl_validation_data.npy"
syn_test_data_path  = synthetic_root / "data" / "ptbxl_test_data.npy"

syn_train_lbl_path = synthetic_root / "labels" / "ptbxl_train_labels.npy"
syn_val_lbl_path   = synthetic_root / "labels" / "ptbxl_validation_labels.npy"
syn_test_lbl_path  = synthetic_root / "labels" / "ptbxl_test_labels.npy"

for p in [
    syn_train_data_path, syn_val_data_path, syn_test_data_path,
    syn_train_lbl_path, syn_val_lbl_path, syn_test_lbl_path
]:
    print(p, "->", p.exists())

/content/Dataset/data/ptbxl_train_data.npy -> True
/content/Dataset/data/ptbxl_validation_data.npy -> True
/content/Dataset/data/ptbxl_test_data.npy -> True
/content/Dataset/labels/ptbxl_train_labels.npy -> True
/content/Dataset/labels/ptbxl_validation_labels.npy -> True
/content/Dataset/labels/ptbxl_test_labels.npy -> True


In [17]:
syn_train_data = np.load(syn_train_data_path)
syn_train_labels = np.load(syn_train_lbl_path)

print("syn_train_data:", syn_train_data.shape, syn_train_data.dtype)
print("syn_train_labels:", syn_train_labels.shape, syn_train_labels.dtype)

syn_train_data: (17441, 12, 1000) float32
syn_train_labels: (17441, 71) float32


### Label mapping + MI target creation

In [24]:
mi_statement_names = [
    "IMI", "ASMI", "ILMI", "AMI", "ALMI",
    "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
    "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]
mi_label_names = [str(label_names[i]) for i in mi_label_indices]

print("MI labels:", mi_label_names)

MI labels: ['IMI', 'ASMI', 'ILMI', 'AMI', 'ALMI', 'INJAS', 'LMI', 'INJAL', 'IPLMI', 'IPMI', 'INJIN', 'PMI', 'INJLA', 'INJIL']


In [25]:
def multilabel_to_binary_mi(y_multihot, mi_indices):
    return (y_multihot[:, mi_indices].sum(axis=1) > 0).astype(np.float32)

syn_train_mi = multilabel_to_binary_mi(syn_train_labels, mi_label_indices)


In [27]:

# optional, if you already loaded these
syn_val_data = np.load(syn_val_data_path)
syn_val_labels = np.load(syn_val_lbl_path)
syn_test_data = np.load(syn_test_data_path)
syn_test_labels = np.load(syn_test_lbl_path)

syn_val_mi = multilabel_to_binary_mi(syn_val_labels, mi_label_indices)
syn_test_mi = multilabel_to_binary_mi(syn_test_labels, mi_label_indices)

print("Synthetic train MI prevalence:", syn_train_mi.mean(), syn_train_mi.sum(), len(syn_train_mi))
print("Synthetic val   MI prevalence:", syn_val_mi.mean(), syn_val_mi.sum(), len(syn_val_mi))
print("Synthetic test  MI prevalence:", syn_test_mi.mean(), syn_test_mi.sum(), len(syn_test_mi))

Synthetic train MI prevalence: 0.25164843 4389.0 17441
Synthetic val   MI prevalence: 0.24806201 544.0 2193
Synthetic test  MI prevalence: 0.25102133 553.0 2203


### Create multihot labels for the real PTB-XL metadata in the same 71-label space

In [ ]:
def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

### Create binary MI targets for the real PTB-XL splits

In [48]:
def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()

df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
    lambda x: multihot_encode(x, len(label_names))
)

df_real["label_mi"] = df_real["label_multihot"].apply(
    lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
)

max_fold_id = df_real.strat_fold.max()

df_train_real = df_real[df_real.strat_fold < max_fold_id - 1].copy()
df_val_real   = df_real[df_real.strat_fold == max_fold_id - 1].copy()
df_test_real  = df_real[df_real.strat_fold == max_fold_id].copy()

print("Real train MI prevalence:", df_train_real["label_mi"].mean(), df_train_real["label_mi"].sum(), len(df_train_real))
print("Real val   MI prevalence:", df_val_real["label_mi"].mean(), df_val_real["label_mi"].sum(), len(df_val_real))
print("Real test  MI prevalence:", df_test_real["label_mi"].mean(), df_test_real["label_mi"].sum(), len(df_test_real))

Real train MI prevalence: 0.2514065908829946 4379.0 17418
Real val   MI prevalence: 0.24736601007787448 540.0 2183
Real test  MI prevalence: 0.2502274795268426 550.0 2198


### Memmap alignment block

In [56]:
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
    df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()
df_memmap_meta["label_mi"] = df_real["label_mi"].values
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values
df_memmap_meta["sex"] = df_real["sex"].values

max_fold_id = df_memmap_meta["strat_fold"].max()

df_train_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] < max_fold_id - 1].copy()
df_val_real_mem   = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem  = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

print(len(df_train_real_mem), len(df_val_real_mem), len(df_test_real_mem))

17418 2183 2198


In [58]:

np.string_ = np.bytes_

In [59]:
input_size = 1000

tfms_ptb_xl = ToTensor()

train_real_ds_full12 = TimeseriesDatasetCrops(
    df_train_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="label_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
    df_val_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="label_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
    df_test_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="label_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

In [61]:
sample = train_real_ds_full12[0]
print(type(sample))
print(sample[0].shape, sample[1])

<class 'clinical_ts.timeseries_utils.tsdata'>
torch.Size([12, 1000]) 0.0


In [64]:
class SyntheticPTBXL8LeadMIDataset(Dataset):
    def __init__(self, signals, labels_binary):
        self.signals = np.asarray(signals, dtype=np.float32)
        self.labels = np.asarray(labels_binary, dtype=np.float32)

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return {
            "signal": torch.tensor(self.signals[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

train_syn_ds = SyntheticPTBXL8LeadMIDataset(syn_train_data, syn_train_mi)

In [71]:
from torch.utils.data import Dataset

class TupleToDictWrapper(Dataset):
    def __init__(self, base_ds):
        self.base_ds = base_ds

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        x, y = self.base_ds[idx]

        x = torch.tensor(x, dtype=torch.float32) if not torch.is_tensor(x) else x.float()
        y = torch.tensor(y, dtype=torch.float32) if not torch.is_tensor(y) else y.float()

        return {"signal": x, "label": y}

In [72]:
train_real_ds = TupleToDictWrapper(train_real_ds_full12)
val_real_ds   = TupleToDictWrapper(val_real_ds_full12)
test_real_ds  = TupleToDictWrapper(test_real_ds_full12)

batch_real = train_real_ds[0]
print(batch_real["signal"].shape, batch_real["label"], batch_real["signal"].dtype)

torch.Size([12, 1000]) tensor(0.) torch.float32


### Create augmented dataset

In [73]:
from torch.utils.data import ConcatDataset

train_aug_ds = ConcatDataset([train_real_ds, train_syn_ds])

print("real train size:", len(train_real_ds))
print("synthetic train size:", len(train_syn_ds))
print("augmented train size:", len(train_aug_ds))

real train size: 17418
synthetic train size: 17441
augmented train size: 34859


In [74]:
from torch.utils.data import DataLoader

batch_size = 64

train_real_loader = DataLoader(
    train_real_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

train_aug_loader = DataLoader(
    train_aug_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_real_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_real_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [76]:
print(df_test_real_mem["sex"].value_counts(dropna=False))

female_test_mask = df_test_real_mem["sex"].astype(str).str.lower().eq("female").to_numpy()
male_test_mask   = df_test_real_mem["sex"].astype(str).str.lower().eq("male").to_numpy()

print("female:", female_test_mask.sum())
print("male:", male_test_mask.sum())
print("test len:", len(test_real_ds))

sex
0    1132
1    1066
Name: count, dtype: int64
female: 0
male: 0
test len: 2198


### Train model on only real ECG data

In [77]:
import torch
import torch.nn as nn

class GRUBinaryMIClassifier(nn.Module):
    def __init__(
        self,
        num_signal_channels=12,
        gru_hidden_dim=128,
        gru_num_layers=2,
        mlp_hidden_dim=128,
        dropout=0.3,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=num_signal_channels,
            hidden_size=gru_hidden_dim,
            num_layers=gru_num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_num_layers > 1 else 0.0,
        )

        pooled_dim = 2 * gru_hidden_dim * 2  # avg + max pooling over bidirectional output

        self.fc1 = nn.Linear(pooled_dim, mlp_hidden_dim)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm1d(mlp_hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(mlp_hidden_dim, 1)

    def forward(self, signal):
        # signal: [B, 12, 1000] -> [B, 1000, 12]
        x = signal.transpose(1, 2)

        # x: [B, T, 2H]
        x, _ = self.gru(x)

        avg_pool = x.mean(dim=1)
        max_pool = x.max(dim=1).values
        feat = torch.cat([avg_pool, max_pool], dim=1)

        x = self.fc1(feat)
        x = self.relu(x)
        x = self.bn(x)
        x = self.dropout(x)
        logits = self.out(x).squeeze(1)

        return logits

In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [82]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    n = 0

    for batch in loader:
        x = batch["signal"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True).float()

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    n = 0

    all_logits = []
    all_probs = []
    all_targets = []

    for batch in loader:
        x = batch["signal"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True).float()

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)

        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

        all_logits.append(logits.detach().cpu())
        all_probs.append(probs.detach().cpu())
        all_targets.append(y.detach().cpu())

    all_logits = torch.cat(all_logits).numpy()
    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()

    return {
        "loss": total_loss / max(n, 1),
        "auroc": roc_auc_score(all_targets, all_probs),
        "auprc": average_precision_score(all_targets, all_probs),
        "probs": all_probs,
        "targets": all_targets,
        "logits": all_logits,
    }

In [80]:
def fit_model(model, train_loader, val_loader, device, epochs=2, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_state = None
    best_val_auroc = -np.inf
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_auroc": val_metrics["auroc"],
            "val_auprc": val_metrics["auprc"],
        })

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_auroc={val_metrics['auroc']:.4f} | "
            f"val_auprc={val_metrics['auprc']:.4f}"
        )

        if val_metrics["auroc"] > best_val_auroc:
            best_val_auroc = val_metrics["auroc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history

In [91]:
real_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

real_model, real_history = fit_model(
    real_model,
    train_real_loader,
    val_loader,
    device,
    epochs=20,
    lr=1e-3,
)

Epoch 1/20 | train_loss=0.4672 | val_loss=0.4793 | val_auroc=0.8636 | val_auprc=0.7211
Epoch 2/20 | train_loss=0.3560 | val_loss=0.3555 | val_auroc=0.8905 | val_auprc=0.7527
Epoch 3/20 | train_loss=0.3143 | val_loss=0.3408 | val_auroc=0.9021 | val_auprc=0.7701
Epoch 4/20 | train_loss=0.2913 | val_loss=2.2040 | val_auroc=0.9014 | val_auprc=0.7656
Epoch 5/20 | train_loss=0.2810 | val_loss=0.3030 | val_auroc=0.9235 | val_auprc=0.8146
Epoch 6/20 | train_loss=0.2670 | val_loss=0.3189 | val_auroc=0.9239 | val_auprc=0.8171
Epoch 7/20 | train_loss=0.2587 | val_loss=0.3256 | val_auroc=0.9150 | val_auprc=0.8054
Epoch 8/20 | train_loss=0.2543 | val_loss=0.3085 | val_auroc=0.9228 | val_auprc=0.8143
Epoch 9/20 | train_loss=0.2518 | val_loss=0.3294 | val_auroc=0.9174 | val_auprc=0.8020
Epoch 10/20 | train_loss=0.2414 | val_loss=0.3559 | val_auroc=0.9284 | val_auprc=0.8223
Epoch 11/20 | train_loss=0.2440 | val_loss=0.3140 | val_auroc=0.9266 | val_auprc=0.8167
Epoch 12/20 | train_loss=0.2343 | val_los

### Train model on augmented ECG data

In [92]:
aug_model = GRUBinaryMIClassifier(num_signal_channels=12).to(device)

aug_model, aug_history = fit_model(
    aug_model,
    train_aug_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-3,
)

Epoch 1/10 | train_loss=0.3577 | val_loss=0.4330 | val_auroc=0.8832 | val_auprc=0.7443
Epoch 2/10 | train_loss=0.2294 | val_loss=0.4145 | val_auroc=0.9118 | val_auprc=0.7973
Epoch 3/10 | train_loss=0.1998 | val_loss=0.3991 | val_auroc=0.9069 | val_auprc=0.7816
Epoch 4/10 | train_loss=0.1884 | val_loss=0.3267 | val_auroc=0.9165 | val_auprc=0.8141
Epoch 5/10 | train_loss=0.1787 | val_loss=0.3029 | val_auroc=0.9213 | val_auprc=0.8236
Epoch 6/10 | train_loss=0.1723 | val_loss=0.3071 | val_auroc=0.9262 | val_auprc=0.8305
Epoch 7/10 | train_loss=0.1666 | val_loss=0.2983 | val_auroc=0.9293 | val_auprc=0.8368
Epoch 8/10 | train_loss=0.1619 | val_loss=0.3065 | val_auroc=0.9286 | val_auprc=0.8331
Epoch 9/10 | train_loss=0.1543 | val_loss=0.3201 | val_auroc=0.9232 | val_auprc=0.8104
Epoch 10/10 | train_loss=0.1507 | val_loss=0.3607 | val_auroc=0.9166 | val_auprc=0.8011


### Comparing models

In [93]:
real_test_metrics = evaluate(real_model, test_loader, device)
aug_test_metrics = evaluate(aug_model, test_loader, device)

print("REAL-ONLY TEST")
print({k: v for k, v in real_test_metrics.items() if k not in ["probs", "targets", "logits"]})

print("\nAUGMENTED TEST")
print({k: v for k, v in aug_test_metrics.items() if k not in ["probs", "targets", "logits"]})

REAL-ONLY TEST
{'loss': 0.2991649013578729, 'auroc': np.float64(0.9285745807590468), 'auprc': np.float64(0.8357462388803866)}

AUGMENTED TEST
{'loss': 0.3128207448507245, 'auroc': np.float64(0.9230880406001765), 'auprc': np.float64(0.8332511721455995)}


### Female / male subgroup evaluation

In [94]:
female_test_mask = (df_test_real_mem["sex"].to_numpy() == 1)
male_test_mask   = (df_test_real_mem["sex"].to_numpy() == 0)

print("female:", female_test_mask.sum())
print("male:", male_test_mask.sum())

female: 1066
male: 1132


In [95]:
def subgroup_metrics(targets, probs, mask):
    t = targets[mask]
    p = probs[mask]
    return {
        "n": len(t),
        "auroc": roc_auc_score(t, p),
        "auprc": average_precision_score(t, p),
        "prevalence": float(np.mean(t)),
    }

print("REAL-ONLY FEMALE")
print(subgroup_metrics(real_test_metrics["targets"], real_test_metrics["probs"], female_test_mask))

print("REAL-ONLY MALE")
print(subgroup_metrics(real_test_metrics["targets"], real_test_metrics["probs"], male_test_mask))

print("\nAUGMENTED FEMALE")
print(subgroup_metrics(aug_test_metrics["targets"], aug_test_metrics["probs"], female_test_mask))

print("AUGMENTED MALE")
print(subgroup_metrics(aug_test_metrics["targets"], aug_test_metrics["probs"], male_test_mask))

REAL-ONLY FEMALE
{'n': 1066, 'auroc': np.float64(0.917837123035262), 'auprc': np.float64(0.8116818055384353), 'prevalence': 0.23733583092689514}
REAL-ONLY MALE
{'n': 1132, 'auroc': np.float64(0.9371680880662917), 'auprc': np.float64(0.8556203327970943), 'prevalence': 0.26236748695373535}

AUGMENTED FEMALE
{'n': 1066, 'auroc': np.float64(0.9042292003947707), 'auprc': np.float64(0.7970245492751618), 'prevalence': 0.23733583092689514}
AUGMENTED MALE
{'n': 1132, 'auroc': np.float64(0.9386640859694753), 'auprc': np.float64(0.8637500734526136), 'prevalence': 0.26236748695373535}
